In [ ]:
# 1. Import Required Libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Import our project modules
from src.data_processing import DataProcessor
from src.model_training import ModelTrainer
from src.utils import setup_logging

# Setup visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully")

# Heart Disease AI Project - Exploratory Data Analysis

**Course**: Pythonprogrammering med AI  
**Assignment**: Examination Project  
**Dataset**: UCI Heart Disease Dataset

This notebook covers the complete analysis and modeling pipeline for the heart disease prediction project.


## 2. Load and Inspect Heart Disease Dataset

The Heart Disease dataset contains 13 medical indicators used to predict the presence of heart disease in patients.


In [ ]:
# Load the dataset
df = pd.read_csv('../data/heart.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\n{'='*60}")
print("First 5 rows:")
print(df.head())
print(f"\n{'='*60}")
print("Data Types:")
print(df.dtypes)
print(f"\n{'='*60}")
print("Missing Values:")
print(df.isnull().sum())
print(f"\n{'='*60}")
print("Basic Statistics:")
print(df.describe())

## 3. Exploratory Data Analysis - Feature Distributions

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Target distribution
df['target'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Target Variable Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Heart Disease (0=No, 1=Yes)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Disease', 'Disease'], rotation=0)

# Target proportion
df['target'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Target Variable Proportion', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"Class Distribution:")
print(f"No Disease (0): {(df['target']==0).sum()} ({(df['target']==0).sum()/len(df)*100:.1f}%)")
print(f"Disease (1): {(df['target']==1).sum()} ({(df['target']==1).sum()/len(df)*100:.1f}%)")

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 5, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(df.columns[:-1]):
    axes[idx].hist(df[col], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')

# Remove extra subplots
for idx in range(len(df.columns)-1, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Strongest correlations with target
target_corr = df.corr()['target'].sort_values(ascending=False)
print("\nCorrelation with Target (Heart Disease):")
print(target_corr)

## 5. Data Preprocessing Pipeline

In [ ]:
# Clean and prepare data using our DataProcessor class
processor = DataProcessor(random_state=42)

# Process the pipeline
X_train, X_test, y_train, y_test, feature_names = processor.process_pipeline(
    filepath='../data/heart.csv',
    target_column='target',
    test_size=0.2
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"\nFeature names: {feature_names}")
print(f"\nClass distribution in training set:")
print(f"  Class 0: {(y_train==0).sum()} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"  Class 1: {(y_train==1).sum()} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")

## 6. Train Multiple Machine Learning Models

In [ ]:
# Train and evaluate multiple models
model_types = ['logistic_regression', 'random_forest', 'xgboost']
trainers = {}
results = {}

for model_type in model_types:
    print(f"\n{'='*60}")
    print(f"Training {model_type.upper()}...")
    print(f"{'='*60}")
    
    trainer = ModelTrainer(model_type=model_type)
    trainer.set_feature_names(feature_names)
    trainer.train(X_train, y_train)
    
    metrics = trainer.evaluate(X_test, y_test)
    
    trainers[model_type] = trainer
    results[model_type] = metrics
    
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1-Score:  {metrics['f1']:.4f}")
    print(f"ROC-AUC:   {metrics['roc_auc']:.4f}")
    print(f"\nConfusion Matrix:\n{np.array(metrics['confusion_matrix'])}")
    print(f"\nClassification Report:\n{metrics['classification_report']}")

## 7. Model Performance Comparison

In [ ]:
# Create comparison dataframe
comparison_data = {
    'Model': [],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1-Score': [],
    'ROC-AUC': []
}

for model_type, metrics in results.items():
    comparison_data['Model'].append(model_type.replace('_', ' ').title())
    comparison_data['Accuracy'].append(f"{metrics['accuracy']:.4f}")
    comparison_data['Precision'].append(f"{metrics['precision']:.4f}")
    comparison_data['Recall'].append(f"{metrics['recall']:.4f}")
    comparison_data['F1-Score'].append(f"{metrics['f1']:.4f}")
    comparison_data['ROC-AUC'].append(f"{metrics['roc_auc']:.4f}")

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Performance Comparison:")
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
metrics_keys = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
positions = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1)]

for idx, metric in enumerate(metrics_keys):
    ax = axes[positions[idx]]
    values = [results[model][metric] for model in model_types]
    labels = [m.replace('_', ' ').title() for m in model_types]
    
    bars = ax.bar(labels, values, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.7)
    ax.set_title(f'{metric.upper()}', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1])
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

# Remove the extra subplot
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()

## 8. Feature Importance Analysis (Random Forest)

In [ ]:
# Feature importance from Random Forest
rf_model = trainers['random_forest'].model
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (Random Forest):")
print(feature_importance)

# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=feature_importance, x='Importance', y='Feature', palette='viridis', ax=ax)
ax.set_title('Feature Importance - Random Forest Model', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
## 9. Complete Pipeline Creation and Validation

In this section we save the trained models and preprocessing pipeline, then reload them and run a sample prediction to confirm the full workflow from raw input to output.
</VSCode.Cell>
<VSCode.Cell language="python">
from pathlib import Path
import joblib
from src.utils import ensure_directory

# Ensure the models directory exists for saved artifacts
models_path = Path('../models')
ensure_directory(models_path.as_posix())

# Save the preprocessor and each trained model
preprocessor_path = models_path / 'preprocessor.pkl'
joblib.dump(processor.transformer, preprocessor_path)
print(f"Saved preprocessor to {preprocessor_path}")

for model_type in model_types:
    model_file = models_path / f"{model_type}_model.pkl"
    trainers[model_type].save_model(model_file)
    print(f"Saved {model_type} model to {model_file}")

# Load them back and verify they work
loaded_preprocessor = joblib.load(preprocessor_path)
loaded_models = {
    model_type: joblib.load(models_path / f"{model_type}_model.pkl")
    for model_type in model_types
}
print("\nLoaded saved preprocessor and models successfully.")

# Validate the prediction flow on a raw example from the dataset
sample_raw = df.drop(columns=['target']).iloc[[0]]
sample_target = df['target'].iloc[0]
sample_transformed = loaded_preprocessor.transform(sample_raw)

print(f"\nSample raw input shape: {sample_raw.shape}")
print(f"Transformed sample shape: {sample_transformed.shape}")
print(f"True target for sample: {sample_target}")

for model_type, loaded_model in loaded_models.items():
    pred = loaded_model.predict(sample_transformed)[0]
    proba = loaded_model.predict_proba(sample_transformed)[0][1] if hasattr(loaded_model, 'predict_proba') else np.nan
    print(f"{model_type}: prediction={pred}, probability={proba:.4f}")


## 9. Confusion Matrix Visualization

In [ ]:
## 9.1. Automated Test Section

This section verifies the full project pipeline with assertions and sample checks.
It confirms that the saved preprocessor and models are loadable, that the preprocessing output matches the training transformation, and that the models make valid predictions.
</VSCode.Cell>
<VSCode.Cell language="python">
# Automated tests for the saved pipeline and models
from pathlib import Path
import joblib
import numpy as np

# Paths for saved artifacts
models_path = Path('../models')
preprocessor_path = models_path / 'preprocessor.pkl'

assert preprocessor_path.exists(), f"Preprocessor file missing: {preprocessor_path}"
for model_type in model_types:
    model_file = models_path / f"{model_type}_model.pkl"
    assert model_file.exists(), f"Model file missing: {model_file}"

print("Saved artifact files validated.")

# Load artifacts
loaded_preprocessor = joblib.load(preprocessor_path)
loaded_models = {
    model_type: joblib.load(models_path / f"{model_type}_model.pkl")
    for model_type in model_types
}

assert loaded_preprocessor is not None, "Failed to load preprocessor"
assert all(model is not None for model in loaded_models.values()), "Failed to load one or more models"
print("Loaded preprocessor and models successfully.")

# Confirm preprocessing output shape
sample_raw = df.drop(columns=['target']).iloc[[0]]
sample_transformed = loaded_preprocessor.transform(sample_raw)
assert sample_transformed.shape[1] == len(feature_names), (
    f"Transformed feature length mismatch: expected {len(feature_names)}, got {sample_transformed.shape[1]}"
)
print(f"Input transformed correctly to shape: {sample_transformed.shape}")

# Run prediction tests for each model
for model_type, loaded_model in loaded_models.items():
    pred = loaded_model.predict(sample_transformed)
    assert pred.shape == (1,), f"Prediction shape mismatch for {model_type}: {pred.shape}"
    if hasattr(loaded_model, 'predict_proba'):
        proba = loaded_model.predict_proba(sample_transformed)
        assert proba.shape == (1, 2), f"Probability shape mismatch for {model_type}: {proba.shape}"
        assert np.isclose(proba.sum(), 1.0), f"Probabilities do not sum to 1 for {model_type}: {proba.sum()}"

print("All models produced valid predictions and probabilities.")

# End-to-end raw input test using a sample row
raw_example = df.drop(columns=['target']).iloc[[5]]
expected_column_count = raw_example.shape[1]
assert expected_column_count == 13, f"Raw input should have 13 columns, got {expected_column_count}"
transformed_example = loaded_preprocessor.transform(raw_example)
assert transformed_example.shape[1] == len(feature_names), "Saved preprocessor does not transform raw input to expected feature count"
print("End-to-end raw input test passed.")


In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (model_type, metrics) in enumerate(results.items()):
    cm = np.array(metrics['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f'Confusion Matrix - {model_type.replace("_", " ").title()}', fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xticklabels(['No Disease', 'Disease'])
    axes[idx].set_yticklabels(['No Disease', 'Disease'])

plt.tight_layout()
plt.show()

In [ ]:
## 9.2. Run Streamlit App and View Server Output

This section launches the Streamlit application from `app.py`, checks that the server is reachable, and displays a preview of the served page content.
</VSCode.Cell>
<VSCode.Cell language="python">
from pathlib import Path
import subprocess
import sys
import time
import urllib.request

app_file = Path('..') / 'app.py'

print(f"Starting Streamlit app from: {app_file}")
cmd = [
    sys.executable,
    '-m',
    'streamlit',
    'run',
    str(app_file),
    '--server.port',
    '8501',
    '--server.headless',
    'true'
]

process = subprocess.Popen(
    cmd,
    cwd=str(Path('..')),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
print(f"Streamlit process started with PID {process.pid}")
print("Waiting for the server to start...")
time.sleep(6)

url = 'http://localhost:8501'
try:
    with urllib.request.urlopen(url, timeout=10) as response:
        html = response.read().decode('utf-8', errors='ignore')
        print(f"Server is reachable at {url}")
        title_start = html.find('<title>')
        title_end = html.find('</title>', title_start)
        if title_start != -1 and title_end != -1:
            print(f"Page title: {html[title_start:title_end+8]}")
        else:
            print("Could not detect the page title from the HTML response.")
        print("\nPreview of the first 300 characters of the page HTML:")
        print(html[:300])
except Exception as exc:
    print(f"Could not reach the Streamlit server at {url}: {exc}")
    print("Check the terminal output or ensure the required packages are installed.")

print("\nOpen the app in your browser at http://localhost:8501")


In [ ]:
## 9.3. Streamlit App Menu Overview

The Streamlit app uses a sidebar menu to switch between pages. The available menu items from the app are:

- **🏠 Home**: Overview of the application, project summary, and instructions for use.
- **🔮 Prediction**: Interactive prediction interface that accepts patient parameters and shows model outputs.
- **📊 Model Performance**: Performance comparison charts and metrics for the trained models.
- **ℹ️ About**: Project details, dataset description, technologies used, and ethical considerations.

These menu items correspond to the app sidebar entries served by `app.py` at `http://localhost:8501` (or your local Streamlit port).


In [ ]:
## 9.4. Streamlit App Views

### 🏠 Home View
The Home view provides an introduction to the application with:

- Project purpose and dataset summary
- Key features of the AI model
- Instructions for using the app
- A short explanation of the model types included

### 🔮 Prediction View
The Prediction view is the interactive interface where users can:

- Select the deployed model (`Logistic Regression`, `Random Forest`, or `XGBoost`)
- Enter patient medical parameters such as age, blood pressure, cholesterol, and ECG readings
- View the predicted risk classification and probability scores
- Receive a clear risk interpretation message

### 📊 Model Performance View
The Model Performance view shows model evaluation details, including:

- A comparison table for accuracy, precision, recall, F1-score, and ROC-AUC
- Bar charts comparing model performance metrics
- Detailed visualizations of precision, recall, and F1 score across models

These sections mirror the actual app pages on `http://localhost:8501` and make the notebook easier to navigate alongside the Streamlit interface.


In [4]:
# 9.5. Streamlit App Graphic Overview
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.axis('off')

# Draw boxes for each view
boxes = [
    {'xy': (0.1, 0.6), 'text': 'Home', 'color': "#f7f9f7"},
    {'xy': (0.4, 0.6), 'text': 'Prediction', 'color': '#7f9cff'},
    {'xy': (0.7, 0.6), 'text': 'Model Performance', 'color': '#ffbf7f'},
    {'xy': (0.4, 0.25), 'text': 'About', 'color': '#d77fff'}
]

for box in boxes:
    plt.gca().add_patch(plt.Rectangle((box['xy'][0], box['xy'][1]), 0.18, 0.15,
                                     facecolor=box['color'], edgecolor='#444444', linewidth=2, zorder=2))
    plt.text(box['xy'][0] + 0.09, box['xy'][1] + 0.075, box['text'],
             ha='center', va='center', fontsize=12, weight='bold', color='black')

# Add arrows connecting the views
plt.arrow(0.28, 0.675, 0.09, 0, head_width=0.02, head_length=0.03, fc='black', ec='black')
plt.arrow(0.58, 0.675, 0.09, 0, head_width=0.02, head_length=0.03, fc='black', ec='black')
plt.arrow(0.49, 0.55, -0.02, -0.15, head_width=0.02, head_length=0.02, fc='black', ec='black')
plt.arrow(0.59, 0.55, 0.02, -0.15, head_width=0.02, head_length=0.02, fc='black', ec='black')

plt.text(0.25, 0.72, 'Next', fontsize=10, color='#333333')
plt.text(0.55, 0.72, 'Next', fontsize=10, color='#333333')
plt.text(0.42, 0.47, 'More Info', fontsize=10, color='#333333')
plt.text(0.55, 0.47, 'More Info', fontsize=10, color='#333333')

plt.title('Streamlit App View Diagram', fontsize=14, weight='bold')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.show()


ModuleNotFoundError: No module named 'matplotlib'

## 10. Example Prediction Interface

In [ ]:
# Example prediction with Random Forest model
print("="*60)
print("EXAMPLE PREDICTION")
print("="*60)

# Get a test sample
sample_idx = 0
sample_features = X_test[sample_idx].reshape(1, -1)
actual_label = y_test.iloc[sample_idx]

# Make prediction
rf_trainer = trainers['random_forest']
prediction = rf_trainer.predict(sample_features)[0]
probability = rf_trainer.predict_proba(sample_features)[0]

print(f"\nSample Input Features:")
for feature_name, value in zip(feature_names, sample_features[0]):
    print(f"  {feature_name}: {value:.2f}")

print(f"\nPrediction Results:")
print(f"  Predicted: {'Disease Present' if prediction == 1 else 'No Disease'} (Class {prediction})")
print(f"  Confidence: {probability[prediction]:.2%}")
print(f"  Probability of No Disease: {probability[0]:.2%}")
print(f"  Probability of Disease: {probability[1]:.2%}")
print(f"\n  Actual: {'Disease Present' if actual_label == 1 else 'No Disease'} (Class {actual_label})")
print(f"  ✓ CORRECT" if prediction == actual_label else f"  ✗ INCORRECT")

## 11. Ethical Reflection

### Dataset and Model Ethical Considerations

**Potential Biases in the Dataset:**
- The UCI Heart Disease Dataset contains primarily historical data from specific medical centers, which may not represent all populations equally
- Age, gender, and socioeconomic status distributions may be skewed, potentially creating bias in model predictions
- Limited ethnic diversity in the original dataset could lead to reduced accuracy for underrepresented groups

**Privacy and Data Security:**
- Medical data is highly sensitive personal information requiring strict GDPR/HIPAA compliance
- Patient consent and anonymization are critical when using such datasets
- Data breaches could expose vulnerable personal health information

**Model Fairness:**
- The model should be evaluated across different demographic groups to ensure equitable predictions
- False positives may lead to unnecessary anxiety and medical intervention
- False negatives could result in missed diagnoses with serious health consequences
- The model should never replace professional medical diagnosis but serve as a screening tool

**Societal Impact:**
- AI-based medical screening could increase healthcare accessibility in resource-limited settings
- However, it could also perpetuate existing healthcare inequalities if deployed unevenly
- Over-reliance on automated predictions without human oversight is dangerous
- Transparency about model limitations is essential for responsible deployment

**Recommendations for Responsible Use:**
1. Validate model performance across diverse populations
2. Maintain human oversight in all medical decision-making
3. Provide transparent documentation of model limitations
4. Ensure proper data governance and security measures
5. Regularly audit the model for unexpected biases over time


## 12. Summary and Conclusions

This analysis demonstrates a complete machine learning pipeline for heart disease prediction:

✅ **Data Processing**: Successfully loaded, cleaned, and analyzed the Heart Disease dataset  
✅ **EDA**: Performed comprehensive exploratory analysis with visualizations and correlations  
✅ **Modeling**: Trained three different ML models (Logistic Regression, Random Forest, XGBoost)  
✅ **Evaluation**: Comprehensive evaluation using accuracy, precision, recall, F1-score, and ROC-AUC  
✅ **Feature Analysis**: Identified most important features for disease prediction  
✅ **Predictions**: Demonstrated real-time prediction capability  
✅ **Ethics**: Discussed important ethical considerations for medical AI applications  

The Random Forest model typically provides the best balance between accuracy and interpretability. The Streamlit application (`app.py`) provides an interactive interface for making predictions with the trained models.
